# **Assignment 1 - Crew Scheduling Problem**


```
The format of these data files is:  
   number of rows, number of columns (n)  
   for each column j (j=1,...,n) in turn:  
      column cost, number of rows covered by j, list of the rows covered by j  
```

### **Problem Statement:**
 + Select a subset of columns to minimize cost while following the constraint that each row must be covered EXACTLY ONCE by the subset.
 + From googling, it looks like the true optimals are: 
 + + `sppnw41: 11307` 
 + + `sppnw42: 7656` 
 + + `sppnw43: 8904`

In [1]:
# Variable Setup

import numpy as np

file_path = 'data/sppnw43.txt'
with open(file_path, 'r') as f:
    data = f.readlines()

data = [[int(x) for x in d.strip().split()] for d in data]

N_ROWS,N_COLS = data[0]

COL_COSTS = np.array([d[0] for d in data[1:]])
COL_ROWS = np.array([np.isin(np.arange(1,N_ROWS+1), d[2:]) for d in data[1:]])

print(f"Num Rows: {N_ROWS} | Num Columns: {N_COLS}")
print(f"Avg Rows per Column: {np.mean([len(rows[rows]) for rows in COL_ROWS]):.2f}")

Num Rows: 18 | Num Columns: 1072
Avg Rows per Column: 4.53


In [2]:
# Getting some idea of scale

total_cost = np.sum(COL_COSTS); avg_cost = np.mean(COL_COSTS)
min_cost = np.min(COL_COSTS); max_cost = np.max(COL_COSTS)
std_cost = np.std(COL_COSTS)

print(f"Total cost: {total_cost} | Avg Cost: {avg_cost:.0f} | STD of Cost: {std_cost:.0f}")
print(f"Min Cost: {min_cost} | Max Cost: {max_cost}")

Total cost: 3402908 | Avg Cost: 3174 | STD of Cost: 1417
Min Cost: 110 | Max Cost: 7130


### **sppnw43 Param Selection**

When designing penalty amounts & temperature, we take into consideration the magnitude of the cost.  

If `Min Cost = 110`, then we say a change in score of 110 is a realistic rough minimum.  
If `Avg Cost = 3174` and `Max Cost = 7130` then we can say a 7500 cost change is a realistic rough maximum given a few swaps around per neighbour.  
For maximum we also need to consider the penalty cost. Lets say this is P, so total is 7500 + P.  

For the penalty, we want to it to be at minimum the cost of the most expensive column. To have it vary over time is best, increasing as we appraoch our final generation.
Lets say minimum is 1,000, maximum is 10,000. Lets say avg 5000.

So maximum change in cost is around 7500 + 5000 = 12500


We want to start with an 80% chance of accepting -12500 difference, so t0 = 56000  
By the end we want a small chance. If we set t_final to 1, e^(-110/1) = 1.69*10^-48. Lets pick t1=8, so e^(-110/8) = 1.07x10^-6

t0 is starting temp,    t1 is final temp

### **sppnw41, sppnw42**

For this we can use the same as sppnw41, as the magnitudes for cost are pretty similar

In [3]:
# Simulated Annealing


# FITNESS FUNCTIONS
def cost(solution: np.ndarray) -> int:
    return np.sum(COL_COSTS[solution])

def zero_penalty(solution: np.ndarray) -> int:
    cols = COL_ROWS[solution]
    row_sums = np.sum(cols, axis=0)
    penalty_rows = row_sums[row_sums < 1]
    return penalty_rows.size # penalty score = number of bad rows

def overlap_penalty(solution: np.ndarray) -> int:
    cols = COL_ROWS[solution]
    row_sums = np.sum(cols, axis=0)
    penalty_rows = row_sums[row_sums > 1]
    return np.sum(penalty_rows - 1) # penalty score = total sum of overlap amount

def fitness(solution: np.ndarray, zero_weight=3000, overlap_weight=2000) -> int:
    return cost(solution) + zero_weight * zero_penalty(solution) + overlap_weight * overlap_penalty(solution)
# --------------------



def neighbour(solution: np.ndarray, k=4) -> np.ndarray:
    # Using inspiration from implementation in powerpoint (slide 15/20)
    # This algorithm uses some repairing to help convergence
    new_solution = solution.copy()

    for _ in range(k): # Do k swaps per neighbour
        row_sums = np.sum(COL_ROWS[new_solution], axis=0)
        row_bad = np.where((row_sums < 1) | (row_sums > 1))[0]

        if row_bad.size: # Repair Sequence for invalid solution
            r = np.random.choice(row_bad)

            if row_sums[r] == 0:
                candidates = np.where(COL_ROWS[:, r])[0]
                # Repair - prefer columns that don't introduce overlaps
                safe = candidates[np.all((COL_ROWS[candidates] == 0) | (row_sums == 0), axis=1)] # safe means for all rows, either the column or row_sum is zero, stopping overlaps
                pick_from = safe if safe.size else candidates
                new_solution[np.random.choice(pick_from)] = True
            else:
                # Repair - remove random overlapping column
                selected_covering = np.where(new_solution & COL_ROWS[:, r])[0]
                new_solution[np.random.choice(selected_covering)] = False
        
        else: # If our solution is valid, remove a random column & then repair
            r = np.random.randint(0, COL_ROWS.shape[1]) # get random row
            selected_covering = np.where(new_solution & COL_ROWS[:, r])[0] # current columns covering that row

            j_off = np.random.choice(selected_covering) # random column to remove
            new_solution[j_off] = False

            ## Repair Sequence - Fix hole made in row with a different column
            candidates = np.where(COL_ROWS[:, r])[0] 
            candidates = candidates[candidates != j_off]
            if candidates.size:
                new_solution[np.random.choice(candidates)] = True
            else:
                new_solution[j_off] = True # If unreplacable, undo the change, as the column is necessary

    return new_solution

def probability(fit_new, fit_old, t):
    if fit_new < fit_old:
        return 1
    return np.exp((fit_old - fit_new) / t)

def simulated_annealing(x0: np.ndarray|None=None, max_iter=10000, t0=1000, t1=1):
    zero_weight_range = [1000,10000]
    overlap_weight_range = [1000,10000]

    def weights(gen):
        zero_w = zero_weight_range[0] + (gen/max_iter) * (zero_weight_range[1] - zero_weight_range[0])
        overlap_w = overlap_weight_range[0] + (gen/max_iter) * (overlap_weight_range[1] - overlap_weight_range[0])
        return zero_w,overlap_w
    
    zero_w,overlap_w = weights(0)

    x = x0 if x0 is not None else np.zeros(N_COLS, dtype=bool); fit = fitness(x, zero_weight=zero_w, overlap_weight=overlap_w)
    best = x.copy(); fit_best = fit

    t = t0; gen = 0

    alpha = np.pow((t1/t0),(1/max_iter)) # t0 is start temp, t1 is end temp

    def temperature(t: float) -> float:
        return t * alpha

    while gen < max_iter:
        t = temperature(t)
        x_new = neighbour(x, k=4)

        zero_w,overlap_w = weights(gen)
        fit_new = fitness(x_new, zero_weight=zero_w, overlap_weight=overlap_w)
        fit_best = fitness(best, zero_weight=zero_w, overlap_weight=overlap_w)

        if probability(fit_new, fit, t) > np.random.rand():
            x = x_new; fit = fit_new
            
        if fit < fit_best:
            best = x.copy(); fit_best = fit

        if gen % 1000 == 0:
            print(f"Current Cost: {cost(x):.0f} - Best: {cost(best):.0f} | Current Fit: {fit:.0f} - Best: {fit_best:.0f} | Gen: {gen} | Temperature: {t:.2f} | Penalty Weights: {zero_w:.0f},{overlap_w:.0f} ")

        gen += 1
    
    return best

x0 = np.random.randint(0,2,N_COLS,dtype=bool)
best = simulated_annealing(x0=x0, max_iter=1000000, t0=56000, t1=8)
best_cols = COL_ROWS[best]
row_sums = np.sum(best_cols, axis=0)

print(f"\nFinal number of columns: {best[best].size} | Overlaps: {row_sums[row_sums > 1].size} | Uncovered: {row_sums[row_sums < 1].size}")
print(row_sums)
print(f"Final Cost: {cost(best)} | Feasible: {'YES' if np.all(row_sums == 1) else 'NO'} | Columns used:", np.where(best)[0])


Current Cost: 1695392 - Best: 1695392 | Current Fit: 4120392 - Best: 4120392 | Gen: 0 | Temperature: 55999.50 | Penalty Weights: 1000,1000 
Current Cost: 22386 - Best: 11220 | Current Fit: 30458 - Best: 12229 | Gen: 1000 | Temperature: 55505.89 | Penalty Weights: 1009,1009 
Current Cost: 15712 - Best: 11220 | Current Fit: 15712 - Best: 12238 | Gen: 2000 | Temperature: 55016.63 | Penalty Weights: 1018,1018 
Current Cost: 19156 - Best: 11220 | Current Fit: 22237 - Best: 12247 | Gen: 3000 | Temperature: 54531.68 | Penalty Weights: 1027,1027 
Current Cost: 16356 - Best: 11220 | Current Fit: 20500 - Best: 12256 | Gen: 4000 | Temperature: 54051.01 | Penalty Weights: 1036,1036 
Current Cost: 20612 - Best: 11220 | Current Fit: 22702 - Best: 12265 | Gen: 5000 | Temperature: 53574.57 | Penalty Weights: 1045,1045 
Current Cost: 17146 - Best: 11220 | Current Fit: 21362 - Best: 12274 | Gen: 6000 | Temperature: 53102.33 | Penalty Weights: 1054,1054 
Current Cost: 19736 - Best: 11220 | Current Fit: 2

KeyboardInterrupt: 